In [1]:
!pip install transformers torch gradio

In [4]:
from transformers import BartForConditionalGeneration, BartTokenizer

# Load model and tokenizer directly 
print("Loading model... this may take a minute the first time ⏳")
model_name = "facebook/bart-large-cnn"
tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)
print("✅ Model loaded!")

# Text
text = """
Nepal is a landlocked country in South Asia, located in the Himalayas. 
It is home to eight of the world's ten tallest mountains, including Mount Everest, 
the highest point on Earth. The country has a rich cultural heritage, with numerous 
temples, monasteries, and festivals. Kathmandu is the capital and largest city. 
Nepal has a diverse geography, ranging from the fertile plains of the Terai 
in the south to the high Himalayas in the north. Tourism, remittances, and 
agriculture are the main pillars of Nepal's economy.
"""

# Tokenize and generate summary
inputs = tokenizer(text, return_tensors="pt", max_length=1024, truncation=True)
summary_ids = model.generate(inputs["input_ids"], max_length=80, min_length=20)
summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("\n📄 Original Text:\n", text.strip())
print("\n✅ Summary:\n", summary)

/opt/anaconda3/envs/dl/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading model... this may take a minute the first time ⏳


Please make sure the generation config includes `forced_bos_token_id=0`. 
Loading weights: 100%|█████████████████████| 511/511 [00:00<00:00, 14912.22it/s]


✅ Model loaded!

📄 Original Text:
 Nepal is a landlocked country in South Asia, located in the Himalayas. 
It is home to eight of the world's ten tallest mountains, including Mount Everest, 
the highest point on Earth. The country has a rich cultural heritage, with numerous 
temples, monasteries, and festivals. Kathmandu is the capital and largest city. 
Nepal has a diverse geography, ranging from the fertile plains of the Terai 
in the south to the high Himalayas in the north. Tourism, remittances, and 
agriculture are the main pillars of Nepal's economy.

✅ Summary:
 Nepal is a landlocked country in South Asia, located in the Himalayas. It is home to eight of the world's ten tallest mountains.


In [5]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
import torch

print("Loading QA model... ⏳")
qa_model_name = "deepset/roberta-base-squad2"
qa_tokenizer = AutoTokenizer.from_pretrained(qa_model_name)
qa_model = AutoModelForQuestionAnswering.from_pretrained(qa_model_name)
print("✅ QA Model loaded!")

def answer_question(question, context):
    inputs = qa_tokenizer(question, context, return_tensors="pt")
    with torch.no_grad():
        outputs = qa_model(**inputs)
    
    start = torch.argmax(outputs.start_logits)
    end = torch.argmax(outputs.end_logits) + 1
    answer = qa_tokenizer.convert_tokens_to_string(
        qa_tokenizer.convert_ids_to_tokens(inputs["input_ids"][0][start:end])
    )
    return answer

# Test
context = """
Nepal is a landlocked country in South Asia, located in the Himalayas. 
It is home to eight of the world's ten tallest mountains, including Mount Everest, 
the highest point on Earth. Kathmandu is the capital and largest city. 
Tourism, remittances, and agriculture are the main pillars of Nepal's economy.
"""

questions = [
    "What is the capital of Nepal?",
    "What is the highest mountain in Nepal?",
    "What are the main pillars of Nepal's economy?"
]

for q in questions:
    print(f"❓ {q}")
    print(f"✅ {answer_question(q, context)}\n")

Loading QA model... ⏳


Loading weights: 100%|██████████████████████| 199/199 [00:00<00:00, 8781.90it/s]
RobertaForQuestionAnswering LOAD REPORT from: deepset/roberta-base-squad2
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ QA Model loaded!
❓ What is the capital of Nepal?
✅  Kathmandu

❓ What is the highest mountain in Nepal?
✅  Mount Everest

❓ What are the main pillars of Nepal's economy?
✅ Tourism, remittances, and agriculture



In [8]:
import gradio as gr

def summarize(text):
    if len(text.split()) < 30:
        return "⚠️ Please enter a longer text (at least 30 words)."
    
    # Clean the text and split into chunks of ~80 words
    text = text.replace('\n', ' ').strip()
    words = text.split()
    chunk_size = 80
    chunks = [' '.join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]
    
    full_summary = []
    for chunk in chunks:
        if len(chunk.split()) < 20:
            continue
        inputs = tokenizer(chunk, return_tensors="pt", max_length=1024, truncation=True)
        summary_ids = model.generate(
            inputs["input_ids"],
            max_length=60,
            min_length=20,
            num_beams=4,
            early_stopping=True
        )
        summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
        full_summary.append(f"• {summary}")
    
    return "\n\n".join(full_summary)

def qa(context, question):
    if not context.strip() or not question.strip():
        return "⚠️ Please provide both a document and a question."
    return answer_question(question, context)

with gr.Blocks(title="AI Text Analysis Tool") as app:
    gr.Markdown("# 🤖 AI Text Analysis Tool")
    gr.Markdown("Built with BART + RoBERTa | BCA Sem 3 Group Project — DWIT")

    with gr.Tab("📄 Summarizer"):
        gr.Markdown("Paste any article or paragraph and get a concise AI-generated summary.")
        sum_input = gr.Textbox(lines=10, label="Paste your text here",
                               placeholder="Paste a news article, paragraph, or any long text...")
        sum_output = gr.Textbox(label="✅ Summary", lines=8)
        sum_btn = gr.Button("Summarize", variant="primary")
        sum_btn.click(summarize, inputs=sum_input, outputs=sum_output)

    with gr.Tab("❓ Question Answering"):
        gr.Markdown("Paste a document, then ask any question about it.")
        qa_context = gr.Textbox(lines=10, label="Paste your document here",
                                placeholder="Paste the text you want to ask questions about...")
        qa_question = gr.Textbox(label="Your Question",
                                 placeholder="e.g. What is the capital of Nepal?")
        qa_output = gr.Textbox(label="✅ Answer", lines=2)
        qa_btn = gr.Button("Get Answer", variant="primary")
        qa_btn.click(qa, inputs=[qa_context, qa_question], outputs=qa_output)

app.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
